# Crawl corpus ViBioMIR trên Kaggle

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; chưa có OCR hoặc browser render JavaScript.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).


## 1. Cấu hình — chỉnh ID tại đây

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên `DELAY`, `ATTEMPTS` và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus.parquet`.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL

SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RUN_TAG = None            # None: tự đặt ids_<ID_FROM>_<ID_TO>; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7'
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}'
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
không ép bản lock Windows của repo. Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot của code trong repo vào output; không tải code từ mạng.
Các file `.py` sinh ra có thể mở để xem. Snapshot có sẵn của run được giữ nguyên khi resume.
Muốn cập nhật snapshot sau khi sửa code repo, chạy `scripts/build_kaggle_notebook.py` để dựng lại notebook.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'crawler_snapshot.py': 'c-qZ<{gd3rb>Q!+(*FRhN(Br#i=Evib<&a}o1$o&C5lui%6F;d7{<<EcA&)oBLK4>TJ8^y%Q@R;U0D|=WmhE?$+}crc5FF`&UY4b=O3uj|G_F#lK+c)@Ad0OH_(`!rRBRQ@e%`czy5mf_514AD=RBsk6(<lSFYc7hIx39tY&F)<jmqED}Co?=DazN@*^h+ccP@|I@4@K|CV_er*XRLWVsWkqv#Nx{gsuKC!ZMS+0+Sw@w}YpQ4l!sbe83%6Q*fahGm?k#gk7w`GkC%@6N)!h`8D?OOj|vg=D>}**q<yyz7jjaX3%PQ9LYF>2x^H^C&HS(VD7sXOc(Z=+!Jst{q0hd6_}|un30Pbe2SAG^zlU(P5dyJF+6bnTBb&8|7SV6qZpLPa|1K{)SnAzi(x!?0z&2<HVnraiV%cqr(b8nFX?<8vgEE@l5@ggvA89qJE2i;Eg|x$`I;?)u&lF8o_uA_2!KtOVy8QSWdW0GkDa-_bPqiA9<LLvT60R>Q^zrK~z7p;a*f$KXDz4G9FgV7f01|Ns|xwP`%N|JWt~xAdwgk^>LmjI7*r-*-?1+Mi!@Cd?;o~T$*L`Y$q$JLN06E##dj7ljv1?ts&X$D9rQh05AZiqyE-Mq?nfxeH}KwLK{E;C4GHUn<{u&l-ii{a2zKgAfX&onT%x5cB3*FCnN09*RS1v?fR{oPTz6&FS<`Y@%j~flmvJ?DxxqSPOc&yM)_LLUxzZcZ@qZyPH^R`*KXcfP}oKAPd))u;>^pT*V%gV2?zemBj{7AP}Wbg15aWtVCt~r$3-^IazGicQ-K)8yD&Zv7pVq`vD7an;l{;hMG1cr9dZo~224m4U~vzTP=Yy#%L$>Oc<X1gDD~WY$L-)8kGYhkK?29EjJu92I^ec69+ufO9tJx{WmHg0T}K}bzJbZC`cb{cB}bMr-5bTZCw>(DJNX>w9oSq^2H76{<T_<EodGDSih~Q%a2gR%AD^7_&e-*j6;b$z4e<zodYu!$oX%Vgj(?EHz$!$qslVE{JenopFp~Y1BNbJU{&}3)&Kq)=mRZ1p{%Ag(6&@6XA*BWK>982aasMk}Qot~gp{HekqvJ;@(yHgq%kk=SZbxf2$-+_9Y7I%K5iC)7+EU{T1jt@J0t=JTw*bbv33nDKuvjn_*Y%M}c-ok9+lV@_2=3j~bpX*F+^+=S1*4l}fKI?`YUA=GH==JomZCKVur6R?k{v|3j)f@_RuaDzc{pqd44(mGG)CxG1>Kupx$1UB83rpC-o2Z{Ai6XZ<E?yWfX^epnD2PGyM4EG?#_F?us1l5B^?IeF@~m!-%XD<yC(vqYEI1ap<k4FJo7p<k%u^u2*?*xLS5)|WSMFfX3$k2BM&xvwVK(yl=O$MefZ7&iN}tIU)LNLw&8dTgze)3!;p#m3dP%~_hAg1#@EAS9$f>5oqO)Bc?ta72@^!OBV->==txMHV4QQ@WoUATlPrTDp_4`j&gv?(>dQ$ivUxs?j4p~-l2eS4=HK*R75fA3JDyPhhA|ua#c%>bK_K=LM3usMs_$&O@dzgtkjNciM?f?J2I7}Tv#0=u9K;}nfc)XrA&lcF8F^T{LxmU5#I?g22vZU4Ljs%=k3b9?Ik-R|xk2kFaLuP_p@tog3NbE0tKp6sR#%qoQ||{I@sAoT)b!JNl7Nr}0vir$HuUKkm^KIt>9F*mOveW~Y<2`F1oMyp>&JHsNAXHr6f9_t=74BJ*vjPcTz{$RI<<wpXNl3{_v1JRp56BKY3KFZH=J-@PBP%5M^0Qg#e6)DhcR$KLi1Uk!PWykxa8~uiQ-P`jA5Wh&Q1j1;YU;sC&<~kJ*cx2!Yd>XSU}*;c|b3KZ&h<hYp;(S>*10>Eh_DcvR)6CI)f=oV%*`h3Ow<MJ2A`<NM$Lo0|3#*74As90>YjmJRt3FOv-Xrd}(WKZTlN*#0)eRr~lVjg|T}*rP0fohgsWlj-mQf`H5Ts`5y?Cpd28F%M-~P)}XP?&i2NjVv>3lu{_(N=D<1Mch(yNE#^~CUrc`=`v>fx<6L&ud%d1zAbNQVbbu;I8#u4?0=3Y94bxbafrb_$tD2<{H4O*rC<J}lx%TEfOjdE%0QQt*LsEx?M3hHJC3(Cvr+0`9SgsIX$H1R(%VHFJa}KH$%rELfK1!Q=(NO_(ngg-(06JZ#$L$_WV(bcxjp`ZGrbbpnL|#Jpf2z~OI(n^U^m2UxiOnqCqWsSUS}ju!<lkp?#lKDeFiVc2k=n*U=-<y^y>VhHUC~3jkxV^`QVWAvM6jL4#0)`t>q<%m{molItzDCp*Nu;L!g~y|!2vBGni(j-Bnil=k!8(dNj!~lH#VlfvEs<A5g;xi)vzoee)q!}i1Z`x825)QHDSc?wj5$lu4se4;=@U8SP{r<7U*PW!0`aDMPz{g?ZWn?%|XXfY0UN7do6puO$7(g!dd#xs3i!#?=Tg5x!62#1yJ5pOQ49M5=LoOyABX>KMAKhqtMB>oSe5=249AGiF`dnT?15zuq<<;0q9XY3g|oF%1uz~0F<pjHk02gIg6^{o=5L-OZKP<`DP)ndkv(EcnsTM1{`X2H6E=Ba7g?76F_;<;V_vO@qW~)q>QnnTE;C3+tQjUqD06oZITG=$3Ua7i+U_bz0h~N0wfGxzP#*;3F{0DveVkBui5drbGg34qy^-1uRf_(O!n}URnsU(v_(_g#sO`c?n@-Q<$yCJ$R2B;PH0C!os^}#)m7S}jure9U*DPKrg1EWx8?L|oL;c-lUfabK0}emlS~sv$n!JM=0kL;qh$M~wP&I2vAi$9khGuR;t8m{i&bi_EKS%D2>DFZf=b6#|6QCQSVrLK(t-ve>NF7ZhEy@7Wd=bN9t`P}KZl3>IXpB!YmF$O6j%ke)`PmVb;C0*m)ei%Ez2VCO+4t3gs_5=Okft(rI@aqrdk<VEQWT{I%lRvWzN<RC=Ku^2utL3XhT9;fJrMNeB)lZ{?&OtzZ0kN;hHo;A)O7gbR6$uiD8z{<^{f^?F(O#o!He!DV7bsfw;pI4K!Y4$$kWTVppvz+jq7!I}$5c1~i-t5WcZ3STRX0u9e5o#^SgUieX~<6kj&FC>bh%W8OvGB^E51X`uKTeamLpa}Y=1ml93rmEaVN42;r^5m<+7qY7<;7GbxE2QU}49f7pqAfKmlPtbS~sO$T2HZKh2S5@ff20D0$TzY$83Q1^eRa~LQ<+6nMx@=<&`vQ0aqyx|6m1C$Yc<2d@c*Uyk01%ropwcE%P&nsNjlWVm(0FCK0yv#U1EQSh1oCUuBUA<CN1?jQuk>pfJ<7dJ=7;Mqt)bdEjSEuEe7CI^Xf5>IJUj@{;)OO+*jzVyJ?C=YD1nL{;%rIe*Fza?)2a{wD0BMdfK(fUW~H5MbcA9DNW1<hLW4Q_9z3Wbc4uLe>)53)w#_08+Wzzg8R-WtLRs7N;UJJg+9y!R(?jTVbO<bHDBAC;#>vl<mH`@3+(Gqz9H+n<#C~s?JTc?EBLe(W%OIzT%OfglDLhWX-2hcTs0ES+RkaiY&I79qgAp8InI#id&T7jxbPMtUf$B0lLY9Q9-DR++4qI@B09DgIvU!Pb2BXg-*t&;+NVMnlX~(fZZf;^4!$3xWW;MBj)}CQsq*kI6w^rEJ2r|6JeyN!6JmoFSIyRSQK!^n5hvdey9Cj()W_{eYOSB0R&_!+>&88>tY7AS^M6M+M-!KV_!Xe*~Af0Q6!)S)K1<5ecG8@1S7neZ*N>nl?2^ts(x?0OHPWrCfF{SdcFG`@uOb^=E`3%)lUqKWwkY$RZM!&v-CbKAS?+J_xyOD%P{dFHCD%iWC=?rjpv+JBgzW^v5MWEi|7S^@!5H1JQp2GYF==cK2{q>6olRpamg1>C`;7dE8JSy{}U}ru8BIrNg)1ZuDu;_It%Y=Y5W{yTgPc$rkmLrgREv_dIyc%>3Qf=^1RXyUqQTxiOoDh9#9}O)9S)pfZpd&@7SlD%~oiGh!cII8LGo2TuvxD~cSysd)h*Hv0ZSZrwJt1q7hF*h)1Yzk|*S@n<|G9P?L8`TWSQ?;?R~0n<?HNE-Xs3g!-YyMF!_x|q_VUD(<UqUaI%Kh6wYRnsXe^r+HnX&5lBW7ykRs4{#8&C%XAjmM8Ce#V9aRh?8T_~yBXi)A^i6aLgrMGX*6U*!$caKY0ghxTiFvIU$7=;B=d~b&ZE0J88B1n-(Ut%$T(;uPlC?&5=OM|rm9P0mHr(^-eM#VD!L6qt9s#YGebEbuS4eOtUG0DN;YEkDI6tDo%{Ift=o-BdUP8GCV7mce9q$2ow2LP3BeJpR7)B=XYQ%NEBmJ48HYj2xn}a>V=KAayT_`{tfjyFXpf|~S2MC>{MO9-W6_4t;bt0$=`CNo-u_AKBe~nw}oL0%7KzF0Okh0aW`#R{Xt5<eGmID%g{R(v9$`Chq(zjtj*CypO>4LyRb_oEnHXDsEz1i>i&v(zQouj9V?n$Q+YSC<5&drx8=4S|s<|G?c(<L2ZUd@jrqMn2)yYiPC(<zDw_h!V~iz5^nG$4?t%I#FnQ7WxF-1!wqR(b8!3`C5Btce)J7BrE_jfJ;{<b7-JD11lujf4#6IoR_rT-fZK2{;T0Y9qfJA&DYby}_rQE3jBcPMJyf8nUKn9|P~CcHlcV@rCd<I<!b@AW80{I}!xjSr(()%H-b2oQZE!{1$487&aEC<IJl~XxA7qRP|tt&1LHM<;n<o;<;X@%_Auz)cklB@S}(Q*}gE??hP>PDA-N1oiJ%-;QD0F(;^&45|G=iBZNW3Ok+WYyNlo@WV$E&zZ5IQYMv7@d?@;f7b76HBu=A#-6}$~Vc@&=4R@2ijPnvMSY*>d0g6*vCPzXtci5>{tBq_AC2Gl=6$@0H8o-<ADSIiUyV@LLJwF`2IS0(q8BCP$>KFy&1fY?#SIvO4N>k%cGpLcJF=}V0V?8X-3@|+kBpAf~+hRc+v6F%3ISFAW1_p5u!seUq>V|0Snob+Q!fTCUC73VnuD*?tEd*zaOdyv2H&9idNM$8N;-f}h=r5X$r4aXQ^RaUzr(p=f(kMGf>xK>LM3F?%%)>6$zcwaIIgfDjtqs7YD+|i0gdX{XEaADXv(alY@>u4v&8UM>vJ^UO%r14(UW2S*vQmr8_XVwh&5qPEF>_GSJ@gx}{91K!&Djv%Jta8Ts@tp_N~Vz_NQbbQc!{LY8W@W6h{7owbTFjgKv|Ot8Tu9$j+}y)c^gjVDe{>zD@jP(9svE=i;gL33W~E+lfVj(Ih)kHXD?pde73V-i9|E!MfeW^>$Dd}#gxjwu+V+mzWf<%n&=Eem*Cia1p{`<Ntimv1n$JC?3qM=WmLGQbf-o|5_h487cLT=v@(@ORgJ-P&OEXvVlU{pSYv4gTQuv7xZ<J#{OO3&VS!yCT-!jU{fZ^#t~6XLL=^=B47==BqBu~|d`Ra9{HtLRU5ApX&<bCQ$t=sY87xM1$jDPMtu5V=Je)Tx-S|WQAhPt@SAlJ9D0*Jlc)sgg>}@VW62WY=!AiY>90a#fm0Z0t21>=L?k&O<`!2FzWb!Ag@2saUXdxnFq^6>u(%$JkTG%V=Aq?)NkIyjG)0wA42Kv0H|JoS5Y7kxL&e2Rh-U7Y@vvEK_J4^H)wU!pN)<m3b6mJHW&D=S!#^1Ju2_4TyBW4Y9f+SW+!qyXo+cqhzYwPF&YS1R#-VG96^C*dcQtRe}a1;njgtQ{(@ot<NQ7rfwRWfv@&{qr%UlX<b^00I@sUEpHX|e5slj4+pRs4Lb#YY<&Dse$Pq8L7LqQN~D{W;k>RyY`>h1WT8Wt@7av%R(ctmf>A_VFMAg()AqqPn}q<5E#eYRK@0+O50zWoOX$%#SVRL|3v*D2hUY%T`tqg}|{7G-3?bf!`#8XttnPmc&C?@Cb<ldM$J=X`2LAnxe>S*d+ukC`+(R{g#_2S%hN2*1LA=^m%99qNkp5H<Pn14KXx`zI$6($f&DLd6LH(?8WJ*@9NkPEEGERk)wELamvE3xL1pt;~s9;7G_ZS2pYDsw)r(iJj-UDTMfhvB-5-IJPSk$1vCkwR>b%SA{^TiU%!2$iyml<yue-Ev~z%@5&?zR!h^*zKWQ07Y+gq35mB_W3O`!qQs@JLKW3ug1}hZDunzG=C@%UM0B{Nun^8eYQzvb0$kkC~G>`%rwcoPg>_od17J9n+cKTM?CLsx`(d{N`2G)BU7o2nSPxtg)?9~6HE@QE=Yt|Pf6}Pv@L(pxirwDRVep?`y;4(=1)?zC9a^Cc~fxaPNT@_zeQ3OB<%k3;XTS^vYc>UwOEoWc2U%K!Bi!WPZZwoT7JsA-U-yeHfJQ)bFT5`8#RJtoU`D!R-Mx<AG>YOY)FvKkm@#yPOAe}{;!hv*!@~*H3uiOsk)C@szlWI%Es3G@QZd@lpc+7Nq66D7~nJ514TW))U#;G#tsKpoyjCP=a7ZvUzB1oo)to>f+G%V7?Ge|nfDP9Hx2(uzSY!xqAtW@rL)ZgLV){;moHYIEgDuKLkvDX`P87M#t$p79aUoRH$13Io41f5f~w-$3itE5<)3lQ*T7!Of^ww;a&nvxv$rjch+r-|WV-1}laDu-MnQ5C||S1in+TJBLT0%$pu)-jslXs;-ch!?%zK!T+dz0DqoFq^#%{IiLF{saDbPQ-l~{N{!v)J3{QwQury>WZy1u<Z)aV3GO28d0}E6j>0)-(nFkWk56sNWqE)1940NQ8^mfg(jFwj*a6fZHt1_j!EMBQBYfT5R_OS9j<iS7tOtlUN%>rfdt9C2c)K$G9KN${M0BL5`MLmT1YoiSkS~Sq#ET{qUO(Jb7{=**Y3P>!$Chh;DVFM*b!2LI&H@!P=JSS)K1%bN-kSiQJnGjszal=#~1CqP)!2XpyA_#4by7&lHP@`b78&L+4@rsW7|eC&8`4NUuZYov@Y~m1a%$@SSRio2?>D4Z1iNg`)haZyy_@$owL!arYIPLdkzM-@jT)C#Tw~7+w1+m9dd=)XBe}v*68CCN~uL1YEg;o+R)kcr#7W-X@Nj$)$mmY>Y0H!@S&?sT!-V~Th(uSI1fSbl+evqTh+g=bL%w$zOw`}d9`350<K^yxux)5d+mmRy2MV6o2AzZwBF&)vf?xHbUn~oMh4gvyY1EW!Rc(RhW|~mtcAZUT2l%gs4xIs_DySh8#8KBLzB4d=7`0cR*T`Ion#n<lASAQ)*5b5^;k0t_4`(9(AH99SM%vBxoMGKp9|HNeJ335Wn;`2u;H9L=XkPvtIj$aL-|<iJdH683%a!Zjf<UDmp4QEV;$d*=ik;b9T#*IjbWK0UgWE6e_oR#7P4BQxkmG*vCKWAY0IQB*%KU(MZ3V)O--WRiC)(&6khe!6KGSWGShEWWY{E(9ax}X48!3Bl>fM7k8L9?OIc01b8J&ILS9I25e2J|7b7IAJsOJ*;kf@E&p6cQWNU2=exy;EWJ7%?mLVI9>`|37+l*&1;|rVCd1>Dd7~d^Z%FWlzgE)hu%A{|Gs@gr5mo1TtL`KqlE<#V!amUJVM3ZeU5&?BvfErZ6J(Yuk2X8w<ZCz}qZ-Z>zszrGbL1vIE8Rgr~+SK@+oP`#4H&c;UEy@*zu_`}LGj#Qj;$Q8tr`nWT{7y#5tev~aE()TA3O17kc7OA10<;C$P$-ns+#(KY$5J%g6mQ9R{>5vvik){{&YDx9Qm39B;IT1Hb39Rwg$hK)%mjsWD}t`^OZ6V-vONk`a5k+$`Dg^hMHVM=Lr-AoM)7#Eqeu}x5+Z9O<d-$RVVDMFWc0LjY}WiK?HXBJP>2p`WhMs}4$!`83@ZopQg4nZucsf$Cz9n#(t3|?jatT#s1EW5e2{hOB9ZJ4(l5JcsW>knnMbwS%^iy?RZCUvRALgfs7ySbXGFgkMl0?0L}bUPA2tmt_s(<mwXi!w<d9@&dxh?D2ek`wtcg`iyFfzHvNRaM;wC!diLw*hCeVnVw)aS4zk$*02=1W#i#xg=!7H|P@kZ|LZT2x-mu<;w23E>jU?Of-7mu}2gGFB4Mef_PJM9+w=VWh}cFeWA*&fTq#tt^=XA>yeS>UVOHZ;+J<wRaRPGt#&eX*}Wn({4r*=9p8)3CFyjxEr`+Ic7->p_<91(IExI$_5V`Bx8zEx23>{G@|%7h2=C+Kk<G)1I(?VQ5wl|7=i@7XG;)^l&-!%S(;?Y3D`;lC|)&sc0UMU-QV3niifB2%QAjuQ-b?eCPAjUrMU`BeYn33eF(8IWIWRo-tk*<Fv@u4=R>YFeyAxg|I2jlF}3lWtO$c&Yu<oEUGE-jzvymi$|)8(xq+fi*qdh@y=w6Qy4a~U2+bGMY%f7M)6pfmMd>@=cK77eerHBbq@Y9y;(HZRembu8OF9zX7E!qg8pv7Teo;1wtc!1RYa}jQ&EtZWQN|^qGUPFgvi0w^Ydm?&ZL^auAZvYqAM#aF9VLtV?^ZR!a3fdLnb>Yc!zl$iL`R4O@t}1mogqE(P}ZDPD9`(lW3k(dS{W(o3U!1l$!8G@aR~eOUJcF@jl^0OlP;&>F);p-K4)4=<h}P`z-z~Iiz4HgAE8GCb5a!6J1{1kAb^*DX29%Mm+5#hf_>nImOWz&LrH&bHpHHC1eEzguUe4!aR-<F2N|qW<sq3@>CS>dGI2SX+!we?5x-WAHa&_iq<oS+M$(jm-k0mt<y~4h7o8`jwr4e=JBkgDJhQ<BD#2-gL;imX{LSNCrU(9p?g{{ss?zufC*5~`Y_$O+o+9eEU81wC#&WIM*`Edu%0T_F{Sk3B*ui;DS@d$5KBxYnp}il`<P3>cHYKlyDHi;%Ox=_%%ie+?9M&M?X=J1G7~3?h>=MeHS+Ch8GkvJT)@$@5buBrJq_bvHs6s)IVpVEEAuGQNd7kZZ!y$P@*~D-w-CuDm_2r#l#UQ0h(btAd4!?_ri9IqEmRcG5Rk;NG5>HzF*$2%=eEBbkNPWei<p7LurkmRyWyb@S();^tT$77-=viJW)PRGv)voW#(dRU8JEDIXe^D!WK`FwZL!1Ob$Gtzo&gQw0TyDX(f}Z)h>|G7?mYsw;^pqm-3RX;IUm3C@we06yT!RJ_@8%obiCf(I9b#Fy>Qa;zVy`HQOC@)%-to(OLW%IFX~y`fwRmgNS&Z<V~CIY%xLYI>Dn`Y?sj~_ky>W=)57kJ9@G(#7bn0C%d+L%j;0`pI+PM1pVEO1LMzMw2ph^ljFMTwiAc4`&jQdDw<LTf2CUGJuV<JYsXG^7SHs^1cG3wl@xDf6V>&lHqQh5|_K$1ON{^yq2&5XOm_EvK%6nO^-`v|Nx+)Sjh0+tXNvxp{;y5#A^cFgO$1t1-LInIol3Eq%Q-Yv?Jzy2H%k9EUt}@k8JPEgD;nB?EB@-Y4z@??oE7tAqbioI-PcRe(3L;sXT-e68L43j-#wm2jTX9yp6&G7AW|)lp%LKgBb>J)gC9v>3s0-#vobI(Y7d~cd#3!(|?JrOAXxwPDnq^Ez;ZYx0Q%jFL#Tls6I#ZB2%|XhiG&eNAcdp-bZd`xx{hQ98y?*7UdzK(fL{Bj@h@Uj`H8Y*)Xl;#Xu10w_3y>#6H>4c22UJkd4nRqFJYA}R#?b0WGzJ9(F?M`oGve|U;hR~rbmhdbc|uPqeo6#<qfF`TxX--$(pOd)*K0`#1paBzERj&^0*)!DRO6aciQkG_Blz^A=p*K+BIU`&cF2?g548!Lg}X$Z+tu!iujoz&T58M$D%4?df)l?|iq_G1U=`tyUBWrc!#C0r@e$FBFHH1Zk(&VuYCAt=*cUiQUF)>zb}lbSmSd6!o&|)7)zt9|O3_PjP-z48_9&pAsKd;s9(@+6;l$u*WK{k{t*0x(+BmeqwpK!+knwL^5Eo&4Yr}|v<8m?z&N;2Z)*@YKo)$oSl&Cc@VmD3nRJpca&Czr~C_jpTPoV7t^uHV!^QGzr8&C;=(+VvFv&rE}L(n7OpTzm<iyNEk8v+FYo30+^aRSK9l*6$2q6NB%1w;gX3z9(}{<nR1<*qyUlDB>LV0Cc5<JKt&YF@9T19B^h`dfzC`!U)Pv%CK6=5rSYD!P$w#30pT7jQ6gJ4O%WogB~-L-%)Kcb;L^YGoZ5tNu{vP$jRR6<PuyS$lT}XAfK3-Pyx`4p;F(Uu3OCD75}uZ?nf-6fq@;)7e}`Xi9w$2NE2FIiDZbg0!>_Y%R(j&;Tu`N7g1CERTnVDM=bpUl(_FyhK&_q~DD!+ndj<=lmKSu3~4DO08WT<V&_&b&}X^pByC_U2G8uk*@D@MmoB3M)DnT5`2lBh!UHQkaGxqxVrg8yr+E2GE?9vQaNso5CxH7QM>DfH5jph(eMaMWHul%05Td#rdjbq0A_H_j*Vefh44#*Tbf)_n^@wMs6-X&^qT^O!ZM{*6_-HhpZHCq8Ocxri8D}1LmzY15uov+;<YLnh)~Ieu<h?=`~H5atLG#Q(?&0^3IqJmkvu|C7+=zD0OEcR4T<zv4_bq5b2iYJzgX9-007CvVh;uB1$Pl4#E_DR5|c-^Ai@D<M~p@>tO@CgBmsc1zDO+Ac3M#Q^6adw0j+|1bk>%HdM1WP3sO&LS1BD<wMf*&+%{%4PU^2Lc3i1$>A-;^Bh*9!&B)i$1~2BM9?~R~Tan9qEYtM?NT`M{brFp|aPX%nbe`L#v$^&2L;_``fg2m@*X}jyI*Rc65{(1w{r-l#kQ;F_O*%tC8%rP80J{J%`6KmE`zm_B5G?lwQCMWw%rIFzqKaQf5@EaaZJ0V(rt9$(<){9(^g&8|EApXWDoLrm#b2C9UaY0AmU?bH-E}(yBVkV?phOa|$P*KYJMe;Wm?Ri=N2IK8G9;vmI@--_wzRWbCR;KR9h1Gnc&#VOrKA_iQ*A5Z(|v0s7vPMqaU$bJ9lf+l3C1-H@sP`NdffmAq&aOX!#U<=jSk$pa%I-8oT#WZQz|t~#cInQZL5UQt<*|2!Ft+X?b5F<E2zi~hVoED*QsrN<l|v8298TJyCWk(8qmy2!#5~-6o~ik-TCH)UeCoK{2~5<7tfE!V{z@-P@X|iW=J~WGG+G#T3<+wKrm2gL}%_W0AwBXZmt6wRw=A?bW@Ix0WGsh$ah&(Em&&eIS(92VO$|rm{*ALX%Ka1f-FlXvp1xR8G0Ut6u7bOtcrpWUh|To{!}%MALL5}NIf8)*ONNJYh+e;CYU<S!D~-1fxZwIGVKZZ6XKnmOa@7q?#@vapq&IgV7ad-1{Xmav-5)EZ)o_V>cX15H3_gi7IZ0^yAtf(?T^3x;HMwI^Wf(XKK$gpPk#OJUp)AWkAL{!`wxEf$-AF?Fpvht7SQ{#3EJQtdO;A1fOTYfE#x2f4bTp~^^G3Qn5v<5Tg>2Q&p28`XsU>>0GUas)2^x95PG*(n~HnB0+#<&O&w8TUVFXuRuC*_74rhG#L!?=!k#Ib5$j^w=_%ToF!A&*DB_t|aU1*7-Ww?q;zSDvPK_M_0>V~Cw$JGd?L7cF-$Me+b|?sd944s27Sn7m;(YpDadQM_Ph%h62=+LN$<@4mjGGc*ANc5}zxm#yU;N<FZ-4jbo$o*T)%(Byw|9}%Jo?UWet-XOKK=e*fBwPuKKtQce)gk(rnf)1|JjfJ{?Xt5?6Z%4_vkxsfBxZbAN{wVeg45u@!O|A{;!|@`0qb|`-4Y+^)3AU+5dj$vtRt=(Fgzb`FDSA+3wxn|Mu69-u|0MKl$0CU;UV1|M#~aefKw?{mYL(|M0^{fBBt9zxpk{{rHDyR(I0L!;gMBb3T6O;r(wrI}iRTojA!HUXhRR>g2&M;YIP_Jy?*PhabJ4IaBziuQV>E7_?k5TEOZR&j_)U&DOQs$>Id$16&jfI!Qiw+3_!2Y=wc@xI{^e$%``+r-_Bt_=H(s6aopISeFf(brTQ9U$lanjmANW0TV$s%xhR^kcVBUXb0R@!O~ZynsJbUAt#c@xF$IY89@PP$cpl!gY}EOQ+Sasr`KeGI(Q+^H5`N{R4POqF2JZN5H(sRRyWza5)VOa&&K6-yrU$I%4>(SB+H}x-n7V8$?Z!|rFgvj)ZJ}pEFRyxOfO!zcNrAG7kCR`%T4RJGje?+=Rtg*&j1lawJhRr&J06fa>_~Te#86H)`i}Mdl!0}_fP;T3P+qDy!QsCOb(rChWz&)R!PZL#^WB{`@?(x;}3uL)8GHEw;%oT$G|W6_osjR^H0C`e;$4J*Y`?{>2vlb4?aXj_wfE-0aZTy=)L%!eD&a;roiJKe)JDV4wdb=mgzIP%wm+34tOp|3dQ{kZHgkqRh#|&D95|1j<Pwq{wMQknBJpQBgi=6b?)ro%_*=PQA!~MZ-o1yVA}+2m}PsQ-`)Ge&wudw2jBYa-EZ9+hO-h7(Oi-CB@sh_|A#sW2q(L0lT%6cE8$aanL+^MV|r4`)VEZ5O{@hvk{Q+CGIAPmQ+f1;xN^YI6prii?UssXtO?s&0yf_N1H`CI&}rRPBBcl+K5QweqB>-an8h4&OW*A=o<wJ#TaxeAn=lxq4heZIn~|#fbvR<4Hp%)0z$cXmFJiE>H1Yr~x*}{DUFev;QC$km7s}whVOYrJ13zYdrFzmNHM1@v28tnU5n7z+#WZDdP$_rI7g5#KT{M>aFfbDVU7FH2u4FL_==Wl~GV_M4xxKY9z%vI55*{+xeM{%4Mmi0ZNh1)~H7}V?qjr^ux+bPC+tsox$OkR2hWJW7=(9!V$P(9YT%uEtrnTh_Xu2e*jvkO|1*W(C#a&6W=qj3}uAq6^xh-Q5DW(+W1Q3Y=M%O05?aA$dGJ`{`lTCsY1WW-j$ap|cN-}lvtz>x%F0_$j(v@eP4aJEXQTENKWOINVZCRap<-;5*3yK;2u;3uI6|Or^<rRB@aag23c8{AUc2BmPV|a1ma;}Gt&MPEnynuj>uYG0{OCKs!*-=_Ad0Tv;R*f~NbouzM#{QPPv0Gk}V_vOgy~0~<UZ>o0hV~{|b-Q*&+F<dml;ToH;!m(M!A3YzcRTPwoJ<@kN5;Ju9c|U_Mx#pt_{M-vbqNN)!A;e?Vz7P-iJ8-`d98@Je5U1=klOt-Ef;`@n`i7-dLV=o8QaSn3-1!>FcVK6W!s`<ly}YGv${0J!3|BHD|J0Mihm_t@!(Qo-3&7he*N0**RJ2XiF=#Sb1j$I1pz$=bg3Y=DKE+s9cY?bU2RTly>l_?e-KtTAXsXPgEq2^acJw@Ik{~bcbV9r%#J|CPJ?Qw#Zlq&FX==O;a8|DQdFiWBY#?ff{Mg$`QzCZ`*f0crwSk77SLO-9Q9r$NMLhEbR)b7r8WIr930R&;nn9oo#{--HGF#u5oL8>J#M7(T0IhaD{{tfM3oW+77`&b?<im6>w1_DFFAO5TL`<#0UlX!#4dpmVz2>mSQy5KWw6I!^p&Dpjl~H`LUuyY^#K_dHCH4hj}vh=l*kz7&&p`{dRg+au)U@)#X-X;$;L+(*)*H1cHtrR!*Jb$dO;U6RMWrgWryTxm(|<uIxNEd$h^?7s$+@A$I>m=8uGWtz18q)HB!YQ7F}*g%~|g@MA3{_iu<Hx&{}lo<HAm1S~<2a8r`w@S9pQdeR0`=L^9+0@e3P(9+UZaoJ1a1>6mxQPs76iFkL@_uW*N-wzu?>j88HQzj04F1wh@Q+iPCUvgF!fG@JwVcvv0JM&W4>g2z8Z3>1{qRnj@7M@00cySj_{&J<&>rLu#kBJ0|Yj^W(#+7V2;H@nNgDI-ccpGI^LHAL=;1cw(abL+4LG|uxO{60hQMeso5Y8){!+k~4^g0x+GuCDxk#@V=t*6W@lJBhz242qY`c87WMdOkIR0Nn#YobJn0CR=jsE}pL(%@cKppKL)Xd@>9R$I!U4kNq0351<r6<~08%$de+@5%Xm#nTzAYzfAhVVF^+a>?OI7+JLK4b_Gw<Qdd@1ZoMWh87b!Ze!LIM4?O}IiQeI^EK(%In3sd{Dvz8HFl!o8fK+niPzp$gZ)*h6Kc9k5g5$3Oc6G&(kOp|2ILAymAY2WFX`7({J~uqSBSQ6HgBTgU0dzysJHL#QL_}QHH8L(hyMwpW%xTWwX`*#x;`c=q?Rh;@Gt{?k@_QV_>E=e=7syc<(_(*Ezq7)AE>x7HnV`rF?iilG%TV~|?>Al!ZoYWq>dQzU)}zOKP_zA<rr?6Nfp36F$$an%t9C=NDLZtgko;VLE|)+jC8o=Fw>v`rp!l6B?!cCll@_kaGcq2g<#L&Mu^_Ar;;NX9F2B2R>*~wFwg2SY(|_N5@!#`&G>3Q;N6F~4Oy+H9cqUXl8+N^Z(~e#CycMS`((5<1qbgVkBQqOnBayt~ONxu>Vp3J7f>$xaKZOFycgEYw+isPYc3PoYnu6w`*dGtSI1%&zpLTpu)Z|)qfmr<}?^&BfOILiginOyf2`)F5VrIDxyAoWhCg?(LO==Zi&T<(y8E}`mjk_+aZY{d&x_(i%w1k?%tZ{X=xElNMZp9w2T+Q-)J|j67bV{D9la=N;M9c~4D#E)aI-JX3l!ir|yQQ1Zmz&!=XdqmCNjD;#lC$}n^nf5D=0ft6nw7Q=H77;KjNBPWdmkGX+hFN-NW&1x04V&qphHGv`=L}fc$!E%WJC~EO;do$^;p%~qzTyD^{G`MOu2DIHH<bC=ZdzEYwnB*+EcP2-!xrNfV>AkILwTcWeT$qE$XMfDVlDCu5f34u*8s{Pbv^NkZn>T2+=CHfP=1&s|##J#Nil|B0O7?;}Ca~91q~nDKwdxUbpH}$5I>!c{T@ZVjNJF<O^FJz#>rUN?^XK_*ro7DFf$h4XD|w2Ei-IrFV`3jH;_~UEx&kg6=k(C&KJbmpIZS^TxDISAz?;8%DnIvKm2hMLXh)CIsS(XHBI;ACzi_cNe3z?M4+INOiWJ8`x5VajVhbVHx#<PJyf2&+(_ywpf$G8<*N6F*ULkr(D{}TAPW*AgdTa0dX0ovovr`8zhDzwS2wbIxLl1;UJ24CxV?}uMW5MDIGZH9C%aVBj6wWR#Q8uq!fvCfC)wk)Y;UsBd$n3i0<TaQdEQvUDcev0}I#a)NL6H?znHPH#Cj!rFYaQ&k=v3Of%MqJZ~F^hrDvq$tM<2cgu0?$Y!r+(uWPGj$zn%HOj7mD|GZ(@*4{rcHp0mR9x`%o+&5(GBCO5=}^@zX+-Typ(#SLfvU$07@)>A)`Y*t^vfKFs4TV+TO&q@L9@dGm>GwFJ_L?^?AXxPjt1~8sCGcH-N|)lzHX@}QGznF1vN5I_yWYIlSLDZB>%vG2VfLm;4PgNLmIjy+0Yr+B=HvWgidv=%hChd(JAyQm`0;GOgTD)PSO>=c;_*<kP-%QjwPYap1&xj1uJb+0?uI+C!YWOMPP3f(96J)53CQCZ31*9K&3#@pJXkUDmxbo3g~t~Ll`<C;(~I9P-L2z9;nJsCvCo0iL*j~_4$w%^FbB&c{_pzM)P5efkvWRPBDq1a?8Q176H>pO`Am`C6hBt=7qz4zTy(6alqV9)BzFA#OH|B!CnHYv@~N1SQ;@RiTI1@{%OG`!^e>oQ9UhRPDbvA!Q3<&^F_C(Nj8l`_W&-I=@}kJx{=-$1;?h&VE5BqA4V5rqBd35np6m9KuJD(Tg<-p*m*2`cQK_N;k&?<7Fb0CY4KH(ZJ%`Pw|R>O>jpUZD5DEN7N)}w7{bBVTFX5)1$8v&;WdaG@=FqhDH$Nl5A6~R()kc@DV<MuBBWaO+>IEbz@#YWpMYFZ(RX#_PYB7PT$jgFKPccn?Y#W(qu-UzoAZYsy)TSJobK!E@r!Zx%JtiHQljgOS5}>?lZW@;PC@t8u=E6P8oLAp^x$2JIsAu`zY|KITylmF{sm|A;r+jZ57N7!#E!v<zfzwCY>KWF01-p4qr{E@fGzU}zZ_1S!@1UEwX<M>hxh-PBAkDcqHh$6%|zU@9Dv#sTDoxc;Ya_NI){J`35HVYJz*Y){YasK*cr{Ck)4P4|BI7k&B81bw2k%7!~6de+PMFp&29oiL4(Q1?>u;)=8tAxL2cRZ23!8dGcE1hoILo~DNfvPITMJS2;Nm;H2V)f`UNPl@x%Kch{be<h!@via&{lye`nZ)>k`=a)5@ffJb1TCZUxIf`Q+CjUIu(b@u$CY_5k>|u^agI;r)LE1o;q{fb+75RGmSg_hT5fC`SGlP<dS2fL;aIpS(wv9=tcJCaGq%6L>}#!Sg(YVa{c^EZ_*N7>G0Q1rI#}-VB{GhQ1=m%qa=AS@m5ZYN$`bJ?B40JCp_R9<JtJ15{kPd(JD=&^_mC6?tG09e3gP6*+hJkl5}ytE;OH{oPvpkE^M(x`3e{NI^=MY;(MF&pAG<IQNMoe(M^Jd{<$Vd{b!<>Xl~9k*_M<2;Qvd{07fmV=uPptl85~!!*FeQ^Je~@5Rh!+;4vKo4lQC&OwFsrqDd-l4CF}`x_lm)wL~;SR-igDlj2IW2KXP^6Q89-%od!Yqxd;&fc!FBQ?V_-9LN%%8l!H{yeyS?bTbi@Ay;NsMHx$ae}I07HkuLFw%aEVzOy=?}Glf3!4uer81*q+1@V8+d_xzmVAqxN4`vss;O5uLus1UrB(})%}ivo>{vy=&dG^#In>?Cjl_(+rbc2$vNB2W^Q7xUX@Rj`l*&Okk87S}&4D8PwT2I6Wo6}R9#IZx3d7+*BjMB%;UskPU5@F8C}#nM0#s>@MOt}GI?9KQuD?_{fyJwC!&Ad+oX_!Cug!r2BZVI}=YY*<FCEeH>;;d7(+jAmB4uy3x`(*?ONA&_Wxp^#jDT1yaL3?;O!r_k-Ym>1M}rnCK>4l0X~fwB^<H&bd|3J^fXTd!6Cd+tVS#FV2(rs613uR#Pzso$L%5z4V=Jm-)JB%<WA^!Og?&bht9lz4ae{qIlwd1|sWTS&yIMDGxj6f@HE>9q1j?q=sMU4Mbo<H{PdVJtro~43t;z|dv=L8p5f<8E*H&Cw+DoW{Iw?JoI9AOXls*Rb%60tRB-`=abBqPeQ`DyC4a58lCk78!SoJx4n>-Wb8Pd;fTjS*|?lzd!0=Nkyzi&O9eTutHTobBBE4xv7*J?NLh{F<fw;bNbX*l5;N}_&HLLcubUOG!G8awE?%h+<=8>W)mva|4%F^S^7AVmwKKz~V$na1~YzR|`3i0KQE%~>#xN-Q#^Zk;7h`WJT=qIpc;elDWDT+0~=+xh=RM;9y};eY?R6xs&*7`YWJ<A1DN3#WC-ntP!o@BN~4B<%=s>Y0+RV|J!rDbtqoF6MSf9j7f(%Wk*m6bD7jiM_dYc4JY*pcX8M0wF3^Y6BDpb}8*D*S(VOl0?iGVPcL<+@e7^><8rR4U|u32mOOo%nwJS0A7e+1=cG2sE)y|jCzDTC91Qa8TqevsH&*vxxkN+cd76k`WSMLJa=_f@J&}8!{}q$rf`iCI0h`a1B5r{RR;Jb5Ikbh(x+yF@mts&WfY9lvT7I)^|aTDC@*@mma={}>%^EQ74#;f-Xuz9efK(M<0|6)2xQT`C@YcMT?{RoS%6(Tl)w}ihtgS!wBYR^^Jrc~t03&T64Y&c!{@><dGMT$43FmtZOr26?J6@z2Vi5c9cLy+HL_}|h{}McsPM+~bl7K1Sb!^J3H)LfnFFQm>)gI#;`j<z8iVkpSM%GkLmg<bwYS+m|0I4ct}RThVs%~;?=UoZ0!vgnTFa+3!pD_X<%y)0L0()iN`{1j)hS@`sM6_R@X`l9`D6$4{1RZ;ce09Sq#mO))9`Q=R7`E^7oe~n3b_D}1@nG9L~b@Jpe=4rY(b|nS*G;mDCo?c6n&7YR8HBAcrT%U5PIQN%A9#N&#fqSol-xbxKt}5sgnxyECn<|5<v90M$wf>NbTI(n!1Q+irEl$A_tEem%vId@l{zVlFe?2uUu_xK8{+CY=r^IbD|^w1p%ec2&4dnF~3DQLe(EPdOSP@Sld(xrCN?V`jb!mUpbAD`~',
 'diagnostics_snapshot.py': 'c-qA~U2o(@a_<Z5f9TU8;BY<StTwloV5AYS*<Ir{@yB{CC&(!Rw}<SRX^TVh_#tVm43NMNMjirWksN~HaESK|!|`1XImZcttRC_p=l+NCOHx(cO*T24k=A!e*06@`kLv3BuIlF2*4D#J@Io-o@;Q&9K1(K(NXCMNl279#FJ!=e_~a>@WXT)`1sOk$SXRb;CgUI~Lm5w5%CkapYR<i_t*zU)<UCEXg7It$eYv=Oi>3!j6p5gaNt`o1;$c$8g~<9W6cb)XMJR)!njYjQdS(bH7IH4kH1pnP`1hkE7S*I#F^{SjnbeQ;6UnhR7X=3Zyz2TePh!1L%8MC{G3z{q*J@ZA@nVu>^Qs|V<kj;>GM&f>(7^v-GYrTY`Uts)F#GJ3XE6{fcZaucG5A+x3;l%uGEtOSOqh64!o!?qc?6^oXGKrML4w2{Ib|`~{(Z-F)nsuNh_qn6ci(@UWeE`Sk9kxIdUhGlvEN$8AmQ;e6>->`IOnuZ%;#b`Walt?>9{Lsoi97~6|LEroUk}4h?1T-lX;Q%+|`{P9;v1*InCLqdOelJOifJy!PQ;kmhu8{m|Gx$DJ(AMW0WJ3T^Tb04I%?E>8as;wqs+V5yjCS43KHB-Mh*o`yZlhh-fhl;koYW6wRfYH><3%|7~W|x!P-+Uz)IpS<Nj_A%DCmMBc;pKAT7pg%zt5X@LAP@!kh2#M3A5_F{30+ZZ_p@M$JNMPU>G^B#eM{s}cyge91WT0T<X`>?el<2ff?A2|{XNhX@$Jo=e|FaadpgM0)uk$s4*Ew|Pj=<1yxi^WiBOIXg+ymx|>g>~b+%!JSLK+4gZJj#WOY?ir<^Mc2LFav>8c>z`jcH_ECz(7Vl-TbJe43_Pyff`8<5SoL4<0{aJt%?TACRY@#ys0Q?cF*-lv_($|_I56(NFT#Cuxh{zKQHHVo-KMw3Ao0|>8i4c6A>4vK$cp<Xf|L@luUC6A7p%z1f0ySM<W~yju8Yf;iQL1e~S+qKI{%R3+VS8Y649hjhI989LD3Y?gz!qXf-{CT!wShFb-<HRjY9BjCn3|XULo~<|kZ6d>jeD*MV6AoBwEb`ETHIg3GTNY>*YRtBapXHkCYP!PUhV9Knlmk{q-7)x}TbE<08e@^Zm4&Wfvx7Xka`#nr_>;PWq74h`%y$&Nw1oTWt#Acn&7aC^rfUlG!D9$LipWoUMpX2~=Yc^5%9@X5?t$E23+stDdhfrwr=X^g;%d<*u%tQK=`vKBzY<t*;zB1jYh*ntE+bVS6{T!g;vGo(XR?3~#H`Dfde=l9TE%RE7c%!{7p1}~f=*Y%2|;E{oBtP6N=kt16Toe>X2w&E6<fpaYcK{8JxQHT&`sHutqRl8rzWbW6y2Zr4>>J-dGSVk*ztM<xfW9T)q)*4hh2E^)S<bmnfXLl8*?@-udYfnwLT94HFK53zat$zy9o{yA7?P{`hXz{df!6YKX#(~O1W>FjfOk^x`*c^0#@<l!5vkUEQE)PMJO7Lj<z0ZEvwO6Q#%L_l^fVW`|%&a;AZcWKyO>G-enMJt|mdFP~T9WBd3lqifu<zf+rAhFyp`vGFy9!5w0^lkTJHREJbv*<KG*@;$=Vv`wycUt|dSh+{d8dX<yst*foD3+g)V-!Lln1t@n~9`hD)ITNKwHe{*T1^D_^i;e{q*Y97cq-xSFb)Vc9nUYC6}MV9Bg4BOawEUUH&{)23TIb`mb26<FUsF4_E-O|B6MIpE88|lR(3dr=Fwnw4%vzDI;KZ7R=;{P<~Fo7!PN+fvR9cj^jYM=+TT;gV9EeY8wHsJ(D1KWf7PNMB7y>?ws>jPLz4519xqpO&{6i4&TX<dz~&=b*;1K;w*)1h>XNJ7+CNn;6R3~Wug^~rBU=C$xv-F@uLz@Qd{U(+rW7kDbmCQm9=MZ(Ni7Qn?}jF=iE`-ahtI@j%m=K730@qwI^3oT5&8$d;w}F8l%-n?MBdd;B@psh77o8YYeKK%f2(J1{|thMJMneIN+$Y@-+$0lShLOp1sB5DM+-02;tQ)$|^!AE`J%%*vaLmNmHmD_qxrY{z9yae5AP4+6BD3CUV5h_L{J8D}6Fn_!A~`E>XhJoPq<>&4nyf3<M^(D06(86-DaTuRkhzB#TAW(ptoHPa;00@xhE|et@YrzEQ9q6iG%en4LHsmb>4u|LrA={ii+<Hh$_O-OY18<M-}=pEd)lRj)M=cez5&Lqm@v63ALGbhC>B(=8j#aUGDg`e+*90uf}J_98~X98@m18%8K$#%3UD$R;R29hD^$1g))RHx(U^Tn~Z`tT3otqp^w^<hJ7i%VZdZ&<nfG?3sc-6`rmmK}*q0r-6I!P~c#*q0jEzQ3^reyY<1yzLFbWKsuEc9U`ve(p}bhmD(0sIfOcj_<S64w3=i>6$)-EmV@XfZ<yO=XigxtB;u~HxV0KIYlE+XW(`K$vc^KjCWF>Z$;z+h0<>Tw{MI`r!)G)Tq8z4XsWB~eS7UjAB?cI}k}PyhRi?mg^jV_lVw^$wv);sb;VX4Q_ga_gdb)nMn!qa}Swd4mIII?|?VaO-J<ml%x(wueW5jk=YXXW~LtBeVmHl1^yglWxjx-yqStB50@2#geU<DcKF$5T3U~C9FWk(^ig0iWIsTieW=NJnIRqldua#qGZ`Y7lG3qbD54(ep+lkxSb=$b>o0|TqLM7z|LA?^x@Bn1v{gwPn_Hp5}fo*~&e#bCs}CgsqwfE4~~CDI-Gtux{`iGv19Q7@Gd)tpuu0!&b57vpMM5PJ=_b+fBOrrZWjO>W<0q0RDf?ShuAyUA*{X;e$aVDevMV@ms5OJHM{b;2g<Lvy{!kf4WaET=6WsAf>kh1wO$fo^Y%g<L+zk{22wbaOZsbg4%sw&5g=W$N0k4Y9Hb0O?#k=^ZN)D{`rhzA+_r++0mJB|dj)Lsu0#0DXO_4%HjuY106c%wt2WgCwQQ?fn=cuNlZbWKyrb#4t3ud<iJ(I5maV&m`kzC<`{@`3%y;(*ed+@U~!QmoEcK-+mbr0La-)0>)y9XC({EW-dl%mWq}bfc&3?q~A6uZN_cX&<Mg-m$@d4?TlQtsNFb@bxqYz6a`v-t1hj@qN_5ECukvP#ps@%uOtaei)zxL5i;$*k`s}gNP!vdp(Qta-vRw97O9}4ul+sz%OW06M_VUyYkv=qyBHET7sY6c%FWP99Nd3sOPDGE><v_xO-pAlFBZ@?PQu0c1oEHZ&i(WZ&%Z!5woADS=uSbs?tL#k+r`6!?U|6%84UC8FAFT;o{vF4MYfGcfXI-cdDuObVKE!-+`aodOUN{fP>&q$-~ve`L-yS;40lr=q8}Lk9;|@0-v;TK#=}TVid~>fAp;)iHt=CffP>_kM9Jy)X~xqbj~Ayik%=X6>HKsCI=!7kAS{OPP9v6@>gPe3fe9F<iA1xw+#9I9+w_<?d$XM`;<5kvi@#&%$N^8}0Z*x<LEhX2Krp8TCEYj5K5R~FMF{TFV?VH}Fz2?PT@KDIhrHZn!R5b_?qB>=4K#jt>Fo{D7Gu->$0o?6n$0&aE<Yo?@p8c`9o%Imx}-6ny6jk9U3@%eg{rB4rE+DkD|A?l)%iKR7pi_ldA7FE10WL({utuOaXGEb8shsU0Ha-wv1$dw(=zYG`J2DF{MmwV$&IlD!n~BG$o0(&jUF^6h|(?=uKtBZ(ELeB_~FQpRi>Zf7CsN43G650_G1ZHNySo;K@(R90#Q29ncdsBi*4)6X&?dj_V+;G_8s;Hv)OG~45jne7z`KYJAg?hI&jGdTUz%_%vnT)8Nlzw-{UWCJ^(5aHqF_js$d&3_<y$l<Pq3{89g3k00e6!v&)yVYJKlv_4p8lO|MZke(>xq^*jKGW~HN*xL$_?R$NbKWahlWB$F#qU4MN>)bBdSbJ(9V$NLcU`noU%sM~XP8FEPnUWeKT9l6cJ5|x#zyxY+z>B^LqbQsHRl4S=;jL;p8514`?+;3O_Tv(kn0!~|7E*cOR;}GSlBS=LlEe;6OZTboibQo7W#BRmi5d49R^8Gz+@b(Wt6J7vwk1v1D{_y>G-$6Nk^@Y-eFTniiVM=iO4Y^e=4<j5UjC+F$`hdJqm4aggG{-9G#BrSOvJTq`7SI?=DiFVYb@7kvd9@J)_Z-s7l2mglDF})$zl^mtK~V=a{xNI>m>PSxZ*8;Z8h0vnJ|8yfdVR26mQeTEdk_0`BnT}gD6~!s2UOa=re3ZVHKmClKY=yPZ4*5i%;H~5_T-T^w5W?S5ge1ZS+Rw{JOxs#hz3bohBaU$%pemVQeFlzEQWq#AWY1rXuh<fvsrTW>OauD;r2@m7Zp_P7c^8|8x3Z^{uOT_#*h{3Osm<nt_LY#t--n%aV{_#`oY1$Q>K&^E3Vo$(J{r7M+Ts-ooR@{tfWEky8xImo6+$C@z0tj;uD!AF{T@a>rWTOECFr;N}ni<{<35<a5Gfxg?L)xVTPhx0Z`cqdI>E-ct}40AeEodPNA;?aJ=e3_kR#lePt@wJ|n#RPoS?7MRLzM!a}h^+HiKigIxnf3<PL*gJ`@AKteA6X^tF(SRQ!^lXgk;6?ngKR-p-*jFY0Z?#^9={FM<k_*J94;taMlCj>AHv!}4P<sT@BMx+s@S+I{D=X0d%KWn;<FF`&6_2+8{1YZ4nyxVG2J~;UAFM*k01}Yk(hG%8mbiA}Q9+u;>c=Qxt4L=kp#Uc)*$onWqpTP1uenBj3m1n)yc34nSgW52_bUm~@6vEIcmR7{yS}en!!3+h0_OI+!JiYuG>>l_#L(TwVySHyCoX8&QV;MEC%@Gj!W{UG13|oR6t<BKwTVtMprMmdrIkM?99UKC-U-JPSGDZ+}fNZ0<YU@{>i;xV1rTtpPM~xd4w84fw^c4AOLLehl2vTW|?zBV5no`Y-7FlS{XK?k}Cc#2hL_EB4NIY!E#J6vOM^a8oA0~P!&(MC%kHG}>)Jr}($moU<=7xT9Os^(!6L<$uRs9SIyvRlsJ##`yjt%{^SIkr6rY&oQ6+<T_Q-!6EMec#;%kc!BIp*debeKoaJW%-%GMQsOrJ~9Ca%rChQoE|Orwb0@%Bht{Q;rMt2UQ1CoN;_j7xOX=C)5*#j$QX(y$)yLe^?`Nbf_QkL}T^drP)|dkFw4p^gX$bIZxHtcaq{wJVvKWbgz$0eKgYC3tc<ZS2(>N%JfY<XsOr=q(G`+sS>Xu;GMQ=?31VdqsMQ)_2A&~Bf6Y82ILxbZ>oJ$qq@F^ah>z*@q<TiKlbKWYEh9<^FEl<zC^eJp%E7$v;HjBv83s7Ul*huECECn+r2Ij%jzJ>>g$K*<XYcAybg{P@igG*_8Y^oVw?sXQ&(P7@N#dk(K_I1?NoKFfWF}oS9|3u5VEc-RA&{lBo}xlqsekOyx%!u<GGg_vh0NX#&xGwt@E#HwWHr2D$0t79oxr}+_(%O6@yOjoOj^Qz3=It)@5J1tqxuiefEgukiE6&yz!)I!wren{R()VY}LD4k~(c|yv$dJx;k4khpBb)d*=?Fp)NO_)~-8iZr&MO$FMb5kymd!G)Y0_QJ}|-sIUqo;JF~r*WR$n@^zzrJ6f$Q>jACSqyGAvyc$X0hy;Mp^%bh|wwssTl&p8};hH@ce*9d1B-q}_7Q;1kwe3mI0tSX{%cKgLn1dBMx@&j^#5`Qt9r`^CRyuB_H}6`m1(hGm)CYk_zktw8J9=vo=KhWkfvB_jl4^ZPVcm^-(o1vy!JPV-%MajpljubB*6f__%3;M6(1cJ1!Y#i@>Z?!u)UiOQ!$mz_U8xwZmlgvMbsnbJ@UdJ`L-ly>r%5i))*)F*js^(oXIrHWI9MF91#~@UYYuD5tcqxDJFi)|DXILlZ|=3_11w>g?YNtvSf+Y=)Pd6G**2X#9)`UeuKF#BwpJ}x;jZdNwNb4fT~{)0vd)Gz%wcJg`5=S23<aH=DhH5ci;e}g6RoZR0r2Z19IFD;3UPE#9pm72Y|0rD>savmD5|S$0RZe1G==yz1@YGnQ`=>t=9*UTZ?-OdnS^;CC15%M_D!9`OBWOoVj-#BsLD$1R;$9)=;|_H4abZ~bQ)-KVAt8fh6zw!%Avv-8UQ-oTHW@#Yp7IbTD^sOINZ5=G+dA3Z|q|=+PWHsh*uXwH!`g)nu?1)#6=T1ts3*B%mRV8)hR7B%_b_z1aeb1Z!j&+dmlV_>&d~N`p+IeefQacN)U~JG}ANZ!3U3?95e@-Z(rK_v}wg^4(2VO)pSt<>wuONHo^Tr2wPQge0_nV&uc1iYu<N$UDR4%Cpy`9%@>I))>nQ0l?!ofJ^<*F>TO%7-3`8N#hy`Jv?0`evnXaNJsVh6*MP1x;(=&gqNkf4kzF3CXOwXyW7vanK#e+>`PNMdYJJHcGwO<_^g(b&l`{2lu=gOF()FOH)YA(^9%M45PX~RxJLLPWJr(`G&-I+1v%L+BcGQh{5Z};7Xw3w@-8GY7BqpeOF8$!_>N}$PhF|mJ1svn)uNCkDdV{8ECSh&M#hg0^i(zAY>pop|6~wx)*tl0$4KPhdOILqL2vY4+3DWoJYcAhMM)<xnG~YZRTW;U_U!24>y#',
 'finalize_snapshot.py': 'c-pO6TW=i6b$%D1|4>WC=$`CMlj3TH^^n7bR?@<XSG$5%YX{&CS~cA@GiCNoeGxfJ5=h{OJO#+Y0Rkg1u(Dwowh_cZ@+3awfeZf+=9lC<r>c8;Zj@wLfH-|Qb<U}CKc{+YYwMFJ4UWXUu8`wW<s!)Caio(;Up$vpsLSUfn5y8Y5K@eFqKl~tMP4N$9ghK~_QZ2_lBGG|j&ohALX=Y_f;7%#P~tbws>0ja+PZl|#{?#1KFMTWsGB#&c^U&Eid0bQG%19IxSv)DIB8Z)zR_88GnK_O(xdr(8An1E{OhYCkeO=U{j5k6OD2;gEvrd9!EFPU%Op(W`RyD*F)NzuD$zk2s!*2l<_(KAIsx$(BMopilX;#V6JwcNe(GWWFDg~?+ZTlF=8apT*Xs!uc)u2{&=A7}LY-{~DK3*zgj!CLv?z4|eIz0NgRe-A6od%1sFzwD3#bUlfsYV}YK-SHbV&l8{>>W#|8iAUdBXDXqEv<jzx4>tFV#uussww55e%HF9QQtQIvv9%VY9k*32+IaBOLE%sZQLMu)HMAV;Sjhl<R!{>3ydwc%fHhkuKdf;$kXYTCs%GYTU!?-4E{?2yd!RLOoFhFw7CK_*@uRQKLw!QVdwz{Mf^N$GnMUqQ_W|*DM}_+!jv#*dy6E7FHYHYM4UCS)>>uaymw8qZv#eyAkgV8q|jt(l?wtL(3HJP3l64pU9|E4`CCz>wL+uyjv0LB9>(^tr;+Q!0KU3WkpAf(_Fx95^)`C>lJF`Pi6+ThlUIs8LkLJ8i>>U(9t1S7PzSL$Qd%RoQN4B=a#j!$^&98dtu4RQKa0K3>d%}b`d@zxOFi&bP07=I6A@KNJEJ=<xXeV@ygWCX8d?A|7-bS3g;yH8qeEEI!qC)lC%^o>UtwjjBnPG&lN&gSdi$A7a5^vhHeg0jagVwh;|aCBiH#`YeThk;WL$t-XL3x<h<FqUDeu~0BocZ*MhRi(S0MsYd8NCp<5^9!;vZ#;iA|gHNtO|G$bB9DE7p$)FpE!qsSzI48C0!wQK@eD3;8yZ3;fcAbNaiH?h387JGz&AW0l74H63sgM>1d4j#OK3=jwXqLgJ-&^vFUN_Y8r2cY9fP6*;~cs?ftLE06_ZaxJ&r!6M>H%h}9c%fwDg^C1}tD>m23tpO)!zG_z#AAa6p<~t9m1|<aMkS56p339JNYHPr=6q|FphG_Hu#7n^5CZ_`r$di@U)kVfAHO3%&#>fO7$UbK33yOUUKYn10eIy6;K_qedn<zH$dd#Vh>eOM{Yd62^y{GIbjVI^<XT-^=8CL{60k>UxhQiiVhyk+v3FqVl80uIL@ZiXnN`0w)~wiKDf=>J4@C|9Ce$y%2Fcpy;+{gXN%LNYbj+f6m0=&u7lmcnY4Y4~l_Y#APv4E?NucuVFC-U43gyC}$=_DmY$RP;+u6WkYfY2jmB&_O8FM@H4VYc6Lq^_S=L4U?{9uYw5b@7MFn+dip;<v*sq&X9<W+`0P&J!_tjrS?!Gb?e6SZbbH4Hh5xgu+=ajY@UISVGX3_Y1;DhXZ75|;bpDvD+;<~nN$8zb950*;~s&hiHAaA$ZOFIJA6ZlMR1%#9o^Chvs8mFsxh+YGf<!>fvk<>bu>bv)UK4n@Tgo3%6+HqF)8F%}|8&NVh;<<i*r0_?RLmt`q~X$-UTksgfo99E3JBJ_qlMSSHIO6I}T&7H43Hw(vSVH%u#e0CBQCm(m*kNbafcKGqFVF%`4bqV~@Cd$A@a$pPTh<$PU!yUGt&63-o<ZEn+SF=~;>P4ktN`4yTzA5UR4YIx^9wnh74AdTsxRrwlRgQ><s6tH&vX6N1#pA9>)Jybt6P2i3Qj+Jl+^w<m5NU5c6?X2Zt-HA;6hW@D@{F#^S)}Mfk8>IG6?zqZy*B7K!9ViCJk5LxeeQ;Rwrh(+n|Mg^&ew^JNTo+{KeM5zTsAbWg4xm<z%`^OZuH-g1NX<%%l}CxBDnl-Q7CF=LI0Dh*RN1iK9+h<9O<izpF`5QWL-?IF1`&&?y0Ct3a`^jYsBe18P3D=Vl~sU?Z-Mm#7)QLBWjxyRX#k)qBK|eSzM$&qZPU;&E@`kUmwC=^!RL_9~_+Rqm4bVjhXd>B_XUgS@fwIB1v;UmibYY`LLs-Wk%N*YE+6iG6+-#OLVTK2BwJG5+d`qi!!gR$Ay&;qAk%2tXz1tyr!u4L^pMLOO8x&%+4d+F<Vx^se(&ix*hZF9I-8hs~94gacygCf_BL<#B@(osn$h!{iZJH@yZ%o46abh`0!gj)(rS8uT*!%Lb=35Kr-=z3N=KJ<yFF&j}QF1Ob#{Dy4h-H4PFGyib8GSQw?XbHdbMOZeSF$oR;BZrmdp%yCku<rO-n6W?Hnjcc;@%pT#1THJP<5b~@rf?7Rh?B1kg@gwDOHOk)nTP|h|YaC>^Q$0DMBiw0b26yh2If(K#3+PLSeK|ig~THB=Hw=HPLwmWq@%EJ)f8lr7QJs#1Si2zR$!Ya-R*8+7>h8I<?d|3oq4?dL$hF$EHQp;esL$8;H2<s>scd924ErN~>)GfmKI+7jnK8N*r<7j2QiGCeM=Kk*HD_!Sxv1rhaMWAz}W`jt^qYweRX-k`d082hkxyLo5L+>g7xOa9@f3thLHM%Yuro_yi?8-ua1F3RVWnnsXG%|y`foKK_yx{iEjumG!6d>DSU<5~9lxRX*5flQCx%_%_TV3ihQuA)w+*15$dfonN-`ri79}&!QNV){fgfNoxw?bTkR@&e3%hca(y`-{(Cs51p)>4Ska0N2*lPVsmyjX}47Ne1v%I8lYDRytiId7Xpqr+CcX9W6JF<4f?W`iXek%FB4_YjrJS*EyoJ=o`Ab|kT2>!sd0*pF2y1@}m*9BeTTJY_W<+#_$Z1NQxG^K3a_I5xvmD||XeuhzePH#?!RHW=(xdJk?t(Mf;zPIj_Kn&?fHo=gGe-8~OMTt~&}2=1fu{%u;AMmiL?f*{z-WJo*f-`>Sii-xWdhrCSc{4~^PMdhqN$#u9#e|n&aC#C8IX;j5Y(Z4;;1^(|%WX3F<L#3z3Q%J8@pxsn`-1m-ineC;>W#cG4=DWU3X2(;MxO-+xciLMz_b9WSf<&74Z~gG@2j}~yFkK^@>Ft9jQ+@UNf2($l27eRf<#!s5#MQ+=RZV!=-v<1CcJTP>^&cydOw7+zU;ZIM-G}gVXV(+=uP%OB0?p+stiSxu6m#X(#UH5zN6~lr>qLwQIsB<B0*20}SFe9#!0ma7`2Jrme?5cd|NJLezdiz=B~k-!@Z#$AZ(~~iI|)wy{HLpnUup6Ex0kPyshH@?SKj`1cHj(G25^ntx8j8=0y{mi=O<o$j8a38z19WNcx&Gp#=*YL8wbrKNc`VF{Hr*{a(5x3nCSv3!Y+Um4U`Q<^e#Kt5A{pI<BWkrCggyEIU0gzd~iSlOeWRU#V-=!&L!EkVnSK{i&)qRNe9aM0AzAy2(^oM=4>7m|6axOvfQvq%fM_+yUoOXR*-Nl9eDnnlVzf-1C~|X+t@SxwIbbWMTn-%23m>^%jve29_am2vTgbzLbW**#UTfxTjIxa2dIyqj4?9nT`}J3iAN8%zk2$aIGSEv{CigGqAyO{#q0U@X{*XT_qG<i&glsnZN98M<NI&Pc7GGJ{qM^k-Rtdscz3D#A^H!k&?f-?PppOci$YeUx?g631BZzl;?!E&;hW0KA^5!(maLi`mfAb3^`%V_YoF90BlstpL9E2zA3b~gSxZiv2rbs4!}B%x>o{dk2(8E)ckZ084Ox5EH;Az&Oqk#S8l=|+<=nq~6(HPP{yuB#Bf0!;CQj&8f|dV~h%84qP9zN&NRe3ZsuWYAoQRksS^-}Y(bdJjQTJ=RV;Ifv@r!b3N~(MPyP4Hsf+bpn6<$Jk|Ail}hcm<hm~P=YtBjseoc)zJ6JHXc_Rh};`iu`#`EMUjhPA2Jt>hRGrzrBy0dvX)KCVpq#?!9p{8}eGrakgFtGi9LHph2J4#I)t`AQA^oOk(imgJ)cd*b<$+t%;t?>`4kfP2{_14Q}P6xNfeto8tgeg0$8j`akaF5oqUjo^k$QRCiM4<0@9pFVu@#nWeA9Bzslad=JnlEX9J+&@rXRv+)0<8AYyKp%4GSr61?87Al1L-B29Uf6E~_)TriW?#z(+de;CINrV`iZWj~;1sCSQo&CO@zwjgJG&okfA#*(&PN|Y&4mtO$&yN&zx(ow&xJWZpeGfyXi$=$@<#+7y1QicIF-)wu};ajefnk&u-k{W#E+R!*%N-diYJVCQqQJ_^HXgjwYNOhZjNJHqOqD^E6rM`>yLg{&R7>a5JSrzM>fZg^?9Uuef??V!#FEvwV?x1SV2VrYD7ALta%%274Z>leQr!O1D*8g1nq#mX)296-J&6re8BA3&N|KMCaXpff5hU;a<#vnw>h>6vd3f=Z|k<<)JHJ)b=YugJHtZ?Fm?BY%T66$NSbCx=WX$tITed+Qv<qO&Kc<b;<)W_xxF1Oj@n4dq`dj@ptZQA8QiL_&{6f8R+|RGtUHtUmxg{zpKPo-ifn(t@fOFDtK6)fNH!l!1XC4D`XWJk=Gr}+V{<ve;}v4q-8*I?Oj@BC0HkbAj_{hkKk(f-+ydaL@|$(`@UcB$Tv5DvRYzt{`-a_TbjfF!^%oFiO<lXCf%&PU@QXTf^9L9Fw<{KX-Nsa>E9&!X6rBU-o1zStt4X`!uR6Ek%4B02$rZ(@l`luu?_m8`Ar=MM&tgxrYA^P?4ggjOT2Yw%is9(-=BR1q@>ZlMiF|e)At<7XN*)e3m0&#sHX?&5`ZO1y$psrAg`C>?>Ut4&w=e`*I~sPZ1bybmCw|rphIxC}gN(Hql%AfqzUxS2%t_2$lv8(Ae=Uj87hDw0jf4rs;s+J&1WJNBw6uqZ1UqE8F{qUpe-TDQa}zuIZ~S6xX;NwutiSZ>*M?cCVd-tgqAO*RS=#>MjWSdddHJiz9J{N(ko4~56E=<~{KgHHIZPuT_BrzS&2hts0|<e6GK7UxBXd{Ks&YVsIP8zIr#)N@O?_rb)9ADs@lvzSHPc+cPgRr+oS%?A%y^Pr){$44ZxC5de2&=u{?Kd0z*wc-itF@x5c?)RR$vXv9t}<DfC?@#fK<Bz(vuM80OfA8wt$t}=Y>12l3>7lxMq(=^UWrI$~#%KTyxCZGVZ@{W2?*=;`Iz0W!zka{6nZ4Z{RYImYF)xm=FS%?~_mXKBpz$Cxi0+`aG?e705>+*zzb%Cu2Q9%yu`80@xUD-uQoBq9~>'}
EMBEDDED_SHA256 = {'crawler_snapshot.py': '4f3e6e74821404f0215be3157a7259b5012e827e54fca31921ee4ec078612ed7',
 'diagnostics_snapshot.py': 'f03a5b21ef02173598b2571b69388f1d43a0695e676fb63340be70238b7b63d4',
 'finalize_snapshot.py': '1d9ec1ce6bdbe2d003b6ae1a3f5106b83f1e27a260f6a11fad8b8c27b1644f21'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if not target.exists():
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
FINALIZER = RUN_DIR / 'finalize_snapshot.py'
# Diagnostics chỉ đọc output, có thể cập nhật riêng mà không đổi crawler/hash resume.
diagnostic_body = zlib.decompress(base64.b85decode(EMBEDDED_FILES['diagnostics_snapshot.py']))
if hashlib.sha256(diagnostic_body).hexdigest() != EMBEDDED_SHA256['diagnostics_snapshot.py']:
    raise ValueError('Snapshot diagnostics bị hỏng.')
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
DIAGNOSTICS.write_bytes(diagnostic_body)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        raise
    finally:
        refresh_diagnostics()
if (RUN_DIR / 'crawl_config.json').exists():
    saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
    if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
        raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
if not requirements_path.exists():
    requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script == CRAWLER else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Crawler tuân thủ robots,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.


In [ ]:
with diagnostic_step('crawl'):
    arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS,
                 '--delay', DELAY, '--attempts', ATTEMPTS]
    if MAX_NEW is not None:
        arguments += ['--max-new', MAX_NEW]
    print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
    run_script(CRAWLER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Corpus accepted chỉ gồm tài liệu qua heuristic; chưa chứng minh chất lượng bằng chấm thủ công.
Audit chỉ chạy khi đủ checkpoint. Nếu đang crawl thử, vẫn xuất phần accepted hiện có;
manifest notebook ghi rõ đây là corpus chưa hoàn tất.


In [ ]:
with diagnostic_step('report_audit_export'):
    run_script(CRAWLER, 'report', '--out', RUN_DIR)
    summary = json.loads((RUN_DIR / 'quality_report.json').read_text(encoding='utf-8'))
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if summary['pending'] == 0:
        run_script(FINALIZER, 'audit', '--out', RUN_DIR)
    else:
        print('Run chưa hoàn tất; bỏ qua audit toàn mẫu. Chạy lại ô crawl để tiếp tục.')
    if summary['accepted'] > 0:
        run_script(FINALIZER, 'export', '--out', RUN_DIR)
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Chưa có tài liệu accepted; xem quarantine.jsonl và QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
from IPython.display import FileLink
import zipfile

refresh_diagnostics(strict=True)
with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
    diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
print('ZIP chẩn đoán trong Output:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint.


In [ ]:
from IPython.display import FileLink

notebook_manifest = {
    'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
    'dataset_revision': DATASET_REVISION, 'source_sha256': SOURCE_SHA256,
    'selected': summary['selected'], 'completed': summary['completed'],
    'pending': summary['pending'], 'accepted': summary['accepted'],
    'complete': summary['pending'] == 0,
    'packages': versions,
    'exported_files': [name for name in ['corpus.parquet', 'corpus.jsonl'] if (RUN_DIR / name).exists()],
}
(RUN_DIR / 'notebook_manifest.json').write_text(json.dumps(notebook_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
refresh_diagnostics(strict=True)
archive = shutil.make_archive(str(WORKDIR / RUN_DIR.name), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('ZIP:', archive)
print('Corpus:', RUN_DIR / 'corpus.parquet')
display(FileLink(str(Path(archive).relative_to(WORKDIR))))
print('ZIP chẩn đoán:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
if (RUN_DIR / 'corpus.parquet').exists():
    display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
    display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))
